# Using the fitted DT to generate scan lines

This notebook loads the model bundle exported from `Fit DT controller parameters to calibration grating scans_v3.ipynb` and uses it to predict trace/retrace scan lines for any user-provided height profile and any combination of experimental controls.

**Prerequisite:** run `python codes/cali_v2_export_models.py` once to write the model bundle to `output/dt_models/calibration_grating_v2/`.

Three predictions are produced per height profile and control setting:

1. **PI fitting** — the DT scanner driven by the ridge-predicted (P, I) gains.
2. **PI + data-driven (hybrid)** — the DT prediction plus a learned residual correction.
3. **Pure data-driven** — a direct regression from controls and FD-prior features to the centred trace/retrace lines (no DT involved).

The hybrid and pure data-driven models are *linear-in-features* so they execute in microseconds; the DT scanner uses the same Numba-JIT'd substep loop as the calibration notebook.

## 1.  Setup

In [ ]:
from pathlib import Path
import sys, json, importlib
import numpy as np
import matplotlib.pyplot as plt
from joblib import load

PROJECT_ROOT = Path(
    '/Users/richardmbp14/Library/CloudStorage/GoogleDrive-yu93liu@gmail.com/'
    'My Drive/Shared/Coding/Digital Twins - SPM'
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BUNDLE_DIR = PROJECT_ROOT / 'output' / 'dt_models' / 'calibration_grating_v2'
assert BUNDLE_DIR.exists(), (
    f'Run python codes/cali_v2_export_models.py first to create {BUNDLE_DIR}'
)
list(BUNDLE_DIR.iterdir())

## 2.  Load the model bundle

In [ ]:
# Stage 1: ridge map (controls -> log10 P, log10 I)
pi_ridge = load(BUNDLE_DIR / 'pi_ridge.joblib')

# Stage 2: data-driven trace/retrace and residual ridges + poly/scaler
dd_pack  = load(BUNDLE_DIR / 'dd_models.joblib')

# FD prior + scanner physical params
fd_prior = np.load(BUNDLE_DIR / 'fd_prior.npz')
scanner_params = json.loads(str(fd_prior['scanner_params']))

# Metadata
meta = json.loads((BUNDLE_DIR / 'metadata.json').read_text())
print('PI ridge features :', meta['feature_columns_pi'])
print('DD ridge features :', meta['feature_columns_dd'])
print('Number of pixels  :', meta['n_pixels'])
print('PI bounds         :', meta['pi_bounds'])
print('Train sizes       :', meta['n_train_pi'], '(PI),', meta['n_train_dd'], '(DD)')

## 3.  Build the DT scanner from the FD prior

In [ ]:
import codes.Scanner_fixed_extended_fd as sfe
import codes.Scanner_numba_substeps_v3 as sn
importlib.reload(sfe); importlib.reload(sn)
sn.install_numba_substep_methods(sfe.ScannerFD, verbose=False)
from codes.dt_static_dynamic_calibration_scaffold_v3 import (
    ScannerRunConfig, run_scanner_line, get_height_from_out,
)

fd_drive_nm = fd_prior['fd_drive_nm']
fd_height   = fd_prior['fd_height']
fd_amp      = fd_prior['fd_amp']
fd_phase    = fd_prior['fd_phase']

measured_fd = {float(fd_drive_nm[i]): {'d': fd_height[i],
                                         'A': fd_amp[i],
                                         'phi': fd_phase[i]}
                for i in range(len(fd_drive_nm))}
surface = sfe.FDDriveSurface.from_measured_fd(measured_fd, **meta['fd_surface_kwargs'])
scanner = sfe.ScannerFD(
    params=scanner_params, conversions={k: 1.0 for k in scanner_params},
    fd_model=sfe.make_normalized_fd_lookup(surface, conv_L=1.0),
)
scanner_cfg = ScannerRunConfig(**{**meta['scanner_cfg'],
                                    'tau_A': None, 'tau_phi': None, 'tau_z': None,
                                    'h_smooth_sigma_px': 0.0, 'use_fast': True})
print('Scanner ready.  A0(min drive)=', scanner.get_A0_hat(float(fd_drive_nm.min())),
       ' A0(max drive)=', scanner.get_A0_hat(float(fd_drive_nm.max())))

## 4.  Helper functions: features, DT run, hybrid + pure data-driven prediction

In [ ]:
PI_FEATURES = meta['feature_columns_pi']
DD_FEATURE_COLS = meta['feature_columns_dd']
PI_BOUNDS = meta['pi_bounds']
TRIM = int(meta['center_trim'])
N_PIX = int(meta['n_pixels'])

FD_TABLE_CACHE = {}
def _fd_table_for(drive_nm):
    k = round(float(drive_nm), 4)
    if k not in FD_TABLE_CACHE:
        FD_TABLE_CACHE[k] = scanner.prepare_fd_table_for_drive(
            float(drive_nm), n_d=scanner_cfg.fd_n_d)
    return FD_TABLE_CACHE[k]

def predict_PI(drive_nm, setpoint, scan_speed_hat, I_gain_exp):
    """Stage-1 ridge prediction.  Returns clipped (P, I)."""
    f = {'log_scan_speed': float(np.log10(max(scan_speed_hat, 1e-12))),
         'drive_nm':       float(drive_nm),
         'setpoint':       float(setpoint),
         'log_I_exp':      float(np.log10(max(I_gain_exp, 1e-12)))}
    feats = np.array([[f[c] for c in PI_FEATURES]], float)
    log_PI = pi_ridge.predict(feats)[0]
    lp = float(np.clip(log_PI[0], PI_BOUNDS['bound_lo_P'], PI_BOUNDS['bound_hi_P']))
    li = float(np.clip(log_PI[1], PI_BOUNDS['bound_lo_I'], PI_BOUNDS['bound_hi_I']))
    return 10.0 ** lp, 10.0 ** li

def dd_feature_vec(drive_nm, setpoint, I_gain_exp):
    """Construct the 10-feature vector used by the data-driven and hybrid models."""
    d  = float(drive_nm); sp = float(setpoint); ig = float(I_gain_exp)
    A0 = float(scanner.get_A0_hat(d))
    log_ig = float(np.log10(max(ig, 1e-12)))
    return np.array([[d, sp, ig, log_ig, A0, np.log10(max(A0, 1e-12)),
                       d*sp, d*log_ig, sp*log_ig, A0*sp]], float)

def _center_line(y, trim=TRIM):
    y = np.asarray(y, float).copy()
    return y - np.nanmean(y[trim:-trim])

def predict_PI_scan_line(h_profile, drive_nm, setpoint,
                          scan_speed_hat=1.0, I_gain_exp=75.0):
    """DT scanner driven by the ridge-predicted (P, I).  Returns (trace, retrace, P, I)."""
    P, I = predict_PI(drive_nm, setpoint, scan_speed_hat, I_gain_exp)
    out = run_scanner_line(scanner, h_profile,
        drive_nm=float(drive_nm), setpoint=float(setpoint),
        scan_speed_hat=float(scan_speed_hat),
        P=float(P), I=float(I), cfg=scanner_cfg,
        fd_table=_fd_table_for(drive_nm))
    sim_tr, sim_rt = get_height_from_out(out)
    return sim_tr[:N_PIX], sim_rt[:N_PIX], P, I

def predict_data_driven(drive_nm, setpoint, I_gain_exp):
    """Pure data-driven prediction: features → centered trace/retrace."""
    X     = dd_feature_vec(drive_nm, setpoint, I_gain_exp)
    Xpoly = dd_pack['scaler'].transform(dd_pack['poly'].transform(X))
    trace   = dd_pack['dd_trace'].predict(Xpoly).ravel()
    retrace = dd_pack['dd_retrace'].predict(Xpoly).ravel()
    return trace, retrace

def predict_hybrid(h_profile, drive_nm, setpoint,
                    scan_speed_hat=1.0, I_gain_exp=75.0):
    """PI + data-driven residual: DT prediction + learned correction."""
    sim_tr, sim_rt, P, I = predict_PI_scan_line(
        h_profile, drive_nm, setpoint, scan_speed_hat, I_gain_exp)
    X     = dd_feature_vec(drive_nm, setpoint, I_gain_exp)
    Xpoly = dd_pack['scaler'].transform(dd_pack['poly'].transform(X))
    res_tr = dd_pack['res_trace'].predict(Xpoly).ravel()
    res_rt = dd_pack['res_retrace'].predict(Xpoly).ravel()
    hyb_tr = _center_line(sim_tr) + res_tr
    hyb_rt = _center_line(sim_rt) + res_rt
    return hyb_tr, hyb_rt, P, I

print('predict_PI_scan_line, predict_data_driven, predict_hybrid ready')

## 5.  Example: synthetic 100-nm calibration grating

First, reproduce the calibration-grating ground truth as the input height profile and predict scan lines under a single set of controls.

In [ ]:
from codes.cali_v2_synthetic_grating import build_synthetic_grating

# 100-nm-plateau grating with the same period as the calibration sample
h_input = build_synthetic_grating(
    n_pix=N_PIX, plateau_height=100.0,
    period_px=64, duty_cycle=0.5, phase_px=1,
    edge_smooth_sigma=1.5,
)

controls = dict(
    drive_nm=30.0,      # nm
    setpoint=0.4,       # fraction of free amplitude
    scan_speed_hat=1.0, # normalised
    I_gain_exp=75.0,    # experimental I-gain
)

tr_PI,  rt_PI,  P, I  = predict_PI_scan_line(h_input, **controls)
tr_dd,  rt_dd         = predict_data_driven(controls['drive_nm'],
                                              controls['setpoint'],
                                              controls['I_gain_exp'])
tr_hyb, rt_hyb, _, _  = predict_hybrid(h_input, **controls)

print(f'Ridge-predicted PI: P={P:.3f}, I={I:.3f}')

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8.0, 5.4), sharex=True,
                        constrained_layout=True)
ax[0].plot(h_input, color='black', lw=1.2, label='input height profile')
ax[0].set_ylabel('height (nm)')
ax[0].set_title('Synthetic 100-nm calibration grating (input)')
ax[0].legend(frameon=False)

off = lambda a: a - np.nanmean(a[TRIM:-TRIM])  # interior-centred
ax[1].plot(off(tr_PI),  color='tab:blue',   lw=1.0, label='PI — trace')
ax[1].plot(off(rt_PI),  color='tab:blue',   lw=0.8, ls=':', label='PI — retrace')
ax[1].plot(off(tr_hyb), color='tab:purple', lw=1.0, label='hybrid — trace')
ax[1].plot(off(rt_hyb), color='tab:purple', lw=0.8, ls=':', label='hybrid — retrace')
ax[1].plot(off(tr_dd),  color='tab:orange', lw=1.0, label='data-driven — trace')
ax[1].plot(off(rt_dd),  color='tab:orange', lw=0.8, ls=':', label='data-driven — retrace')
ax[1].set_xlabel('pixel')
ax[1].set_ylabel('height (nm, centered)')
ax[1].set_title(f'Predicted scan lines  (d={controls["drive_nm"]:.0f} nm, '
                 f'sp={controls["setpoint"]:.2f}, '
                 f'Ig$_\\mathrm{{exp}}$={controls["I_gain_exp"]:.0f})')
ax[1].legend(frameon=False, ncol=3, fontsize=8)
fig

## 6.  Example: custom height profile

The DT works on any 1-D height profile, not just the calibration grating.  Build a profile with a steep ascending step (50 nm), a flat region, and a chirped sinusoid; predict scan lines under the same controls.

In [ ]:
x = np.arange(N_PIX, dtype=float)
h_custom = np.zeros(N_PIX, dtype=float)
h_custom[64:128]  = 50.0                         # 50-nm plateau
h_custom[128:192] = 50.0 + 30.0 * np.sin(2 * np.pi * (x[128:192] - 128) / 16.0 *
                                          np.linspace(1, 2.5, 64))
from scipy.ndimage import gaussian_filter1d
h_custom = gaussian_filter1d(h_custom, 1.0)      # soften step edges
h_custom -= h_custom.min()

tr_PI,  rt_PI,  P, I  = predict_PI_scan_line(h_custom, **controls)
tr_dd,  rt_dd         = predict_data_driven(controls['drive_nm'],
                                              controls['setpoint'],
                                              controls['I_gain_exp'])
tr_hyb, rt_hyb, _, _  = predict_hybrid(h_custom, **controls)

fig, ax = plt.subplots(2, 1, figsize=(8.0, 5.4), sharex=True,
                        constrained_layout=True)
ax[0].plot(h_custom, color='black', lw=1.2)
ax[0].set_ylabel('height (nm)')
ax[0].set_title('Custom height profile (step + chirped sine)')

ax[1].plot(off(tr_PI),  color='tab:blue',   lw=1.0, label='PI')
ax[1].plot(off(tr_hyb), color='tab:purple', lw=1.0, label='hybrid')
ax[1].plot(off(tr_dd),  color='tab:orange', lw=1.0, label='data-driven')
ax[1].set_xlabel('pixel'); ax[1].set_ylabel('predicted trace (nm)')
ax[1].set_title(f'Predicted forward scan  (P*={P:.3f}, I*={I:.3f})')
ax[1].legend(frameon=False)
fig

**Important caveat for the data-driven and hybrid models.**  The data-driven Ridge regressions were trained on the calibration-grating dataset and learn *centred* scan-line shapes that mirror the periodic grating.  When applied to a profile with very different spatial-frequency content (such as the chirped sine above), the data-driven and hybrid outputs reproduce the structure they learned on the training data, not the structure of the new input.  The PI prediction, by contrast, runs the actual DT scanner on the new input and therefore correctly follows the new geometry.

Use the data-driven and hybrid branches when the new control settings stay close to the calibration distribution and the topography is similar in spectral content to the training samples.  For topographies far outside the calibration data, the PI branch is the only model that remains physically meaningful.

## 7.  Sweeping controls

The DT runs in milliseconds, so a full grid sweep over controls is interactive.  Below: predicted trace–retrace alignment (the scan-quality score) over a 6×8 grid of (drive, I-gain) at a fixed setpoint and scan speed.

In [ ]:
drives  = np.linspace(10, 100, 8)        # drive amplitudes (nm)
i_gains = np.linspace(40, 110, 6)        # experimental I-gains
Q = np.zeros((len(i_gains), len(drives)), dtype=float)

for di, d in enumerate(drives):
    for gi, ig in enumerate(i_gains):
        tr, rt, P, I = predict_PI_scan_line(
            h_input, drive_nm=d, setpoint=0.4,
            scan_speed_hat=1.0, I_gain_exp=ig,
        )
        t = tr[TRIM:-TRIM] - np.nanmean(tr[TRIM:-TRIM])
        r = rt[TRIM:-TRIM] - np.nanmean(rt[TRIM:-TRIM])
        Q[gi, di] = float(np.sqrt(np.nanmean((t - r) ** 2)))

fig, ax = plt.subplots(figsize=(7.2, 4.0), constrained_layout=True)
im = ax.pcolormesh(drives, i_gains, Q, shading='auto', cmap='viridis_r')
cb = fig.colorbar(im, ax=ax)
cb.set_label('DT-predicted trace–retrace RMSE (nm)')
ax.set_xlabel('drive amplitude (nm)')
ax.set_ylabel('I-gain (experimental units)')
ax.set_title('Predicted scan quality on the 100-nm grating (sp = 0.4)')
fig

Darker cells are predicted to give cleaner scans (lower trace–retrace RMSE).  This is exactly the lookup operators can use to pre-select controls before acquiring a real scan.

## 8.  Standalone replot from a previously saved bundle

All three prediction functions are defined entirely in terms of the bundle files under `output/dt_models/calibration_grating_v2/` plus the project's `codes/` modules.  To reuse this notebook elsewhere, only those two locations need to travel with the bundle.

If only fast-path data-driven and hybrid predictions are needed (no DT execution), `predict_data_driven` requires nothing from `codes/Scanner_*`; only `pi_ridge.joblib`, `dd_models.joblib`, and the FD prior for the `scanner.get_A0_hat(d)` feature are necessary.